# AgriTwin ML Pipeline: 04 - Custom CNN Baseline Classifier

## 1. Custom 4-Block CNN Architecture
Baseline deep convolutional neural network built from scratch for tomato leaf disease classification.

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models

def build_custom_cnn(num_classes=10):
    inputs = layers.Input(shape=(224, 224, 3))
    x = layers.Rescaling(1./255)(inputs)
    
    # Block 1
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, 2)(x)
    
    # Block 2
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, 2)(x)
    
    # Block 3
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, 2)(x)
    
    # Block 4
    x = layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2, 2)(x)
    
    # Classifier Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="Custom_CNN_Baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_custom_cnn()
model.summary()

## 2. Training with Callbacks & Checkpointing

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint("../models/custom_cnn.keras", save_best_only=True, monitor="val_loss"),
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2, monitor="val_loss")
]
# history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)